# Reproducing Chartrand et al. (2023) from dandiset 000630

**Paper.** Chartrand T, Dalley R, Close J, et al. (2023). Morphoelectric and transcriptomic
divergence of the layer 1 interneuron repertoire in human versus mouse neocortex. *Science* 382,
eadf0805. [doi:10.1126/science.adf0805](https://doi.org/10.1126/science.adf0805). The claims file
was written from the author manuscript in PubMed Central (PMC11864503) with its supplementary
Materials and Methods, and from the paper's code,
[AllenInstitute/patchseq_human_L1](https://github.com/AllenInstitute/patchseq_human_L1) at commit
7777857c1b89b19735e45fd9c9bd7493cc1d805c, which this entry reads for the cell labels.

**Dandiset.** [DANDI:000630](https://dandiarchive.org/dandiset/000630/0.230915.2257), version
0.230915.2257, "Human L1 patch-seq electrophysiology": 210 NWB files, one human cell each. This is
the version the paper cites (claims file, opening comment) and the only published version; the
draft has the same 210 assets. The mouse comparison data (DANDI:000020) are used in
`mouse_comparison.ipynb`.

**What is tested here.** The files hold the current clamp recordings only. The features are
computed here with IPFX from the long square sweeps (reading (a) of the claims file's open detail
O1); features that rest on code the paper did not release (irregularity ratio, depolarizing sag,
ADP, the ramp and short square features, and others) come from the paper's per-cell table
(reading (b)). Every analysis is run on both. Cell types come from the paper's cell table, linked
to the files by the claim `dandiset-cell-link`.

| Claim | Notebook | Status |
|---|---|---|
| dandiset-cell-link (added link claim) | this | tested |
| patchseq-counts | this | tested |
| fig2c-subclass-ephys-differences, fig2b-subclass-ephys-profiles, figs3a-sag-adaptation, fig2c-lamp5-gradient | this | tested |
| figs6-donor-condition-ephys, figs6c-region-ephys | this | tested |
| fig4b-mc4r-sag-irregularity, fig4b-adarb2-irregular-firing, fig5b-tnfaip8l3-ephys | this | tested |
| fig6a-ephys-dprime-subclass-structure, fig6d-gmm-subclass-f1, figs8a-classifier-cv | this | tested |
| figs8b-classifier-holdout | none | blocked: the Amsterdam and Szeged recordings are not in the dandiset |
| mouse-cell-link (added link claim), fig3d-ephys-species, fig3g-variability-dimensions, fig4d-mouse-no-rosehip-ephys, fig5d-mouse-pax6-doublets | mouse_comparison.ipynb | tested |
| fig3e-tea-validation, fig3f-a-type-conductance | none | blocked: data not public or not in the files |
| fig6b-dprime-correlation | none | blocked: not attempted in this run |
| fig2d-subclass-morphology, fig4b-mc4r-axon, fig5b-tnfaip8l3-dendrites, figs6a-tumor-dendrite-branching (`data: elsewhere`) | morphology_tables.ipynb | tested on the paper's deposited morphology features |
| fig3b-l1-cytoarchitecture (`data: elsewhere`) | none | blocked: the code rescales the mouse values with scanner resolutions that are not public |
| 13 claims on transcriptomic references, mouse morphology and morphology-based models (`data: elsewhere`) | none | blocked: not attempted in this run |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import l1ephys as L

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")

started = time.time()
meta = L.version_metadata()
assets = L.list_assets()
print(f"dandiset {L.DANDISET_ID} version {L.DANDISET_VERSION}: {meta['name']!r}, "
      f"{len(assets)} assets, {assets['size'].sum() / 1e9:.2f} GB")
feats = L.all_cell_features(assets, workers=12)
print(f"features extracted from {len(feats)} files in {time.time() - started:.0f} s")
labels = L.load_labels()
paper_raw = L.load_paper_features()
print(f"paper's tables (commit {L.LABELS_COMMIT[:7]}): {len(labels)} labelled cells, "
      f"{len(paper_raw)} cells in aibs_features_E.csv")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
s = meta["assetsSummary"]
print("Archive summary: files", s["numberOfFiles"], "| cells", s["numberOfCells"], "| subjects",
      s["numberOfSubjects"], "| species", sorted({x["name"] for x in s["species"]}))
print("Files read:", len(feats), "| errors:", int(feats.get("error", pd.Series(dtype=str)).notna().sum()))
print("Distinct subject_id:", feats.subject_id.nunique(), "| distinct session_id:", feats.session_id.nunique(),
      "| distinct cell names:", feats.cell_name.nunique())
print("Species in files:", feats.species.value_counts().to_dict())
print("Institution:", feats.institution.value_counts().to_dict())
print("Sex:", feats.sex.value_counts().to_dict())
age_y = feats.age.str.extract(r"P([\d.]+)D")[0].astype(float) / 365.25
print(f"Age: {age_y.min():.1f} to {age_y.max():.1f} years (stated as P<days>D)")
print("Files per subject: ", feats.subject_id.value_counts().describe()[["min", "50%", "max"]].to_dict())

The archive's summary counts 210 files, 210 cells and 53 subjects, all human, which the files
confirm. Each file holds one cell. `subject_id` is a LIMS donor id, and ages are stated in days.
The species is spelled two ways ("Homo Sapiens" and "Homo sapiens"). The two files of donor
922967568 state an age of P0.0D (0 years); the paper's table gives that donor's age as unknown, so
0 stands for a missing value. One file states sex "unknown" where the table gives M.

In [ ]:
# Stimulus codes present, and the sweeps used for features
codes = feats.codes.str.split(";").explode().str.split(":").str[0].str.replace("_DA_0", "")
print("Files holding each stimulus code (top 30):")
print(codes.value_counts().head(30).to_string())
print("\nLong square sweeps per file:", feats.n_long_square.describe()[["min", "50%", "max"]].to_dict())
print("Sweeps failing the QC applied here, per file:", feats.n_sweeps_qc_failed.describe()[["min", "50%", "max"]].to_dict())
print("Sweeps used per file:", feats.n_sweeps_used.describe()[["min", "50%", "max"]].to_dict())
print("Long square sweeps stored with zeros before the end of the stimulus (left out):",
      int(feats.n_sweeps_zero_padded.sum()), "in", int((feats.n_sweeps_zero_padded > 0).sum()), "files")
print("Stimulus onset (s):", feats.stim_start_s.value_counts().to_dict())
print("Stimulus duration (s):", feats.stim_dur_s.round(3).value_counts().to_dict())

The long squares are 1 s steps (Methods), starting 0.6 s or 1.02 s into the sweep depending on
the protocol version. Sweeps that stopped early hold zeros (0 mV) for the samples not acquired, where
IPFX expects missing values; IPFX's integrity check then passes them and a step to 0 mV is read as
a spike. They are left out here, as counted above. Files also hold short squares, ramps and in some cells chirps and
nucleated-patch voltage clamp protocols (`NucVC...`), which are not used here.

In [ ]:
# Units. Voltages are stored with conversion to volts and currents to amperes; the values read
# here are in mV and pA. bias_current is stated in amperes in the NWB standard.
import h5py, io, requests
r = requests.get(assets.url[0], timeout=300)
f = h5py.File(io.BytesIO(r.content), "r")
k = [k for k in f["acquisition"] if "bias_current" in f["acquisition"][k]][0]
g = f["acquisition"][k]
print("data unit:", g["data"].attrs["unit"], "conversion:", g["data"].attrs["conversion"],
      "| IGOR units:", list(g["data"].attrs.get("IGORWaveUnits", [])))
print("bias_current value:", float(np.ravel(g["bias_current"][()])[0]),
      "| attrs:", {a: g["bias_current"].attrs[a] for a in g["bias_current"].attrs})
f.close()
print("\nVoltage ranges of features computed here: v_baseline", feats.v_baseline.min().round(1), "to",
      feats.v_baseline.max().round(1), "mV; rheobase", feats.rheobase_i.min(), "to", feats.rheobase_i.max(), "pA")

The bias current is stored in pA, with a `unit` attribute of "pA", where the NWB schema fixes
the unit of `bias_current` as amperes; a reader that follows the schema gets values 10^12 times
too large. The QC here reads it as pA. Membrane potentials (-80 to -50 mV baselines) and stimulus amplitudes
are in the expected ranges.

In [ ]:
# Clock: the hyperpolarizing response starts at the stimulus onset in the same series.
print("Median voltage change from 100 ms before onset to 100-200 ms after it, hyperpolarizing sweeps (mV):")
print(feats[["clock_dv_true_mv", "clock_dv_shifted_mv"]].describe().round(2).to_string())
print("Cells with a negative change at the true onset:", int((feats.clock_dv_true_mv < -1).sum()), "of",
      int(feats.clock_dv_true_mv.notna().sum()))

Stimulus and response are in the same file with the same sampling rate, and the response follows
the stimulus onset: the hyperpolarizing response appears at the true onset and not with the onset
taken 0.3 s early. This supports the shared clock of stimulus and response within each sweep.

In [ ]:
# Recordings deposited twice: identical feature vectors or identical session start times.
cols = ["sag", "input_resistance", "tau", "v_baseline", "rheobase_i", "threshold_v_rheo", "upstroke_rheo"]
vec = feats[cols].round(6).fillna(-999).apply(tuple, axis=1)
print("Files sharing all of", cols, ":", int(vec.duplicated(keep=False).sum()))
print("Files sharing a session start time:", int(feats.session_start.duplicated(keep=False).sum()))

No two files share their features or their start time; no recording is deposited twice.

## The dataset's cells and the paper's cell table (link claim `dandiset-cell-link`)

> The dandiset's files can be matched one to one to the cells of the paper's per-cell table
> (human_l1_dataset_2023_02_06.csv), and the matched records agree on what both describe.

The files carry no t-type or subclass. Every claim that compares cell types takes the labels
from the paper's table, so they depend on this link.

In [ ]:
feats["session_id_int"] = feats.session_id.astype("int64")
by_roi = labels.reset_index().dropna(subset=["ephys_roi_id"]).astype({"ephys_roi_id": "int64"}).set_index("ephys_roi_id")
linked = feats.join(by_roi, on="session_id_int", rsuffix="_table")
print("Files matched by session_id = Ephys_Roi_Result id of the table's nwb_path:",
      int(linked.spec_id.notna().sum()), "of", len(linked))
same_name = linked.cell_name == linked.cell_name_table
print("Of these, cell name in the file equal to the table's cell_name:", int(same_name.sum()))
print(linked.loc[~same_name, ["path", "cell_name", "cell_name_table"]].to_string(index=False))
print("Distinct table cells matched:", linked.spec_id.nunique())
sex_ok = linked.sex.str[0] == linked.donor_sex.astype(str).str[0]
print("Sex agrees with the table's donor_sex:", int(sex_ok.sum()), "of", int(linked.donor_sex.notna().sum()))
age_file = np.floor(linked.age.str.extract(r"P([\d.]+)D")[0].astype(float) / 365.25 + 0.05)
age_tab = linked.donor_age.str.extract(r"(\d+)")[0].astype(float)
print("Age in years (file, rounded) equal to the table's donor_age:", int((age_file == age_tab).sum()), "of",
      int(age_tab.notna().sum()), "| largest difference:", float((age_file - age_tab).abs().max()))
print("Donors per subject_id:", linked.groupby("subject_id").donor_name.nunique().max())

In [ ]:
# Features computed here against the paper's per-cell values (reading (a) against (b) of O1)
computed = L.with_adapt_order(linked.set_index("spec_id"))
fcols = L.feature_columns(paper_raw)
cmp_rows = []
for c in [c for c in fcols if c in computed.columns]:
    a, b = computed[c].astype(float), paper_raw[c].reindex(computed.index).astype(float)
    ok = a.notna() & b.notna()
    close = (np.abs(a[ok] - b[ok]) <= 0.01 * np.abs(b[ok]) + 1e-9)
    cmp_rows.append({"feature": c, "n_both": int(ok.sum()), "only_here": int((a.notna() & b.isna()).sum()),
                     "only_paper": int((a.isna() & b.notna()).sum()),
                     "within_1pct": round(close.mean(), 3) if ok.any() else np.nan,
                     "spearman": round(stats.spearmanr(a[ok], b[ok])[0], 3) if ok.sum() > 2 else np.nan})
cmp = pd.DataFrame(cmp_rows).set_index("feature")
print(cmp.sort_values("within_1pct").to_string())
# The order of the adaptation ratios (O1): which order matches the paper's table
for name in ["upstroke", "downstroke", "width", "isi"]:
    b = paper_raw[f"{name}_adapt_ratio"].reindex(computed.index)
    for order in ["3over1", "1over3"]:
        a = computed[f"{name}_adapt_ratio_{order}"]
        ok = a.notna() & b.notna()
        print(f"{name}_adapt_ratio, order {order}: median |a/b - 1| = {np.median(np.abs(a[ok] / b[ok] - 1)):.3f}, "
              f"spearman {stats.spearmanr(a[ok], b[ok])[0]:.3f} (n={ok.sum()})")

In [ ]:
# Reading (a): a feature computed here is used when it agrees with the paper's value for most
# cells (within 1% for at least half of them); otherwise, and for every feature not computed
# here, the paper's value is used (reading (b)).
COMPUTED = [c for c in cmp.index if cmp.loc[c, "within_1pct"] >= 0.5]
NOT_USED = [c for c in cmp.index if c not in COMPUTED]
print(f"{len(fcols)} features in the paper's table; computed here and used: {len(COMPUTED)}; "
      f"computed but not agreeing (paper's value used): {NOT_USED}")
print("Features taken from the paper's table:", len(fcols) - len(COMPUTED))
hyb = L.apply_code_filters(L.hybrid_table(computed, paper_raw, COMPUTED))
pap = L.apply_code_filters(paper_raw)
print("Cells in the paper's table not in the dandiset:", len(paper_raw.index.difference(computed.index)))

**Result: reproduced.** All 210 files match a cell of the paper's table through the LIMS
identifier (`session_id` equals the `Ephys_Roi_Result` number in the table's `nwb_path`), one
to one. 204 files carry the table's cell name; 6 carry a time stamp in its place. Sex and age agree
with the table for every cell, and the features computed here from the files agree with the
paper's own values within 1% for 71 to 98% of cells, feature by feature, and with Spearman
correlations of 0.92 to 1.00. The adaptation ratios agree less (within 1% for 4 to 53% of cells;
the third over the first spike is the order that matches) and are taken from the paper's table,
with the 55 features computed by code the paper did not release.

In [ ]:
# Sets used below (claims file, "Sets used below"; l1_load.py)
cells = linked.set_index("spec_id")
cells["homology_type"] = pd.Categorical(cells["t-type"].map(L.HOMOLOGY),
                                        categories=["LAMP5", "MC4R", "PAX6", "L1 VIP", "other"], ordered=True)
cells["l1_type"] = cells.homology_type.notna()
cells["core_l1_type"] = cells.homology_type.isin(L.SUBCLASSES)
cells["failed_fx_long_squares"] = paper_raw["failed_fx_long_squares"].reindex(cells.index)
cells["P"] = cells.in_strict_set & (cells.collaborator == "AIBS") & (cells.failed_fx_long_squares == False)
print("Dandiset cells:", len(cells), "| in the strict L1 set:", int(cells.in_strict_set.sum()),
      "| AIBS:", int((cells.collaborator == "AIBS").sum()), "| in P:", int(cells.P.sum()))
lab_P = labels[labels.in_strict_set & (labels.collaborator == "AIBS")].join(paper_raw[["failed_fx_long_squares"]])
lab_P = lab_P[lab_P.failed_fx_long_squares == False]
print("P in the paper's table:", len(lab_P), "| of which in the dandiset:", int(lab_P.index.isin(cells.index).sum()))
print("Strict L1 set in the paper's table by site:", labels[labels.in_strict_set].collaborator.value_counts().to_dict())
print("Cells by t-type in P:")
print(cells[cells.P]["t-type"].value_counts().to_string())
print("Donors:", cells.donor_name.nunique(), "| cells per donor:", cells.donor_name.value_counts().describe()[["min", "50%", "max"]].to_dict())

def table(kind):
    # Cells with their labels and the features of reading (a) (hybrid) or (b) (paper)
    feat = hyb if kind == "a" else pap
    return cells.drop(columns=[c for c in fcols if c in cells.columns]).join(feat[fcols])
TA, TB = table("a"), table("b")

# Features of the Amsterdam (Huib) and Szeged (Gabor) cells from the paper's code. They are not
# in the dandiset and are used only where a claim's method needs the other sites (stated there).
tamas = L.apply_code_filters(L.read_table("tamas_features_E.csv", index_col=0))
mansv = L.apply_code_filters(L.read_table("mansvelder_features_E.csv", index_col=0))
sites = pd.concat([
    labels[labels.collaborator == "Gabor"].join(tamas, on="sample_id"),
    labels[labels.collaborator == "Huib"].join(mansv, on="sample_id"),
    labels[labels.collaborator == "AIBS"].join(hyb)], sort=False)
sites["homology_type"] = pd.Categorical(sites["t-type"].map(L.HOMOLOGY), categories=L.SUBCLASSES + ["other"])

The dandiset holds exactly the paper's primary set P: the 210 Seattle (AIBS) cells of the strict
L1 set whose long square features were extracted. The 62 Amsterdam and Szeged cells of the strict
set are not in it.

## patchseq-counts

> The human L1 patch-seq dataset has 250 cells passing transcriptomic quality, 194 with
> electrophysiology passing quality control and 71 reconstructed; by subclass, 96 LAMP5, 35 MC4R,
> 33 PAX6, 15 L1 VIP and 15 other cells have electrophysiology.

In [ ]:
P = cells[cells.P]
counts = P[P.l1_type].homology_type.value_counts().reindex(["LAMP5", "MC4R", "PAX6", "L1 VIP", "other"])
print("Cells of P in the ten L1 t-types:", int(P.l1_type.sum()), "(expected 194 +/- 19.4)")
print(pd.DataFrame({"count": counts, "expected": [96, 35, 33, 15, 15]}).to_string())
print("Dandiset files not in P:", len(cells) - len(P), "| cells of P without an L1 t-type:", int((~P.l1_type).sum()),
      P[~P.l1_type]["t-type"].value_counts().to_dict())
print("By donor: cells of P in L1 t-types per donor, median",
      P[P.l1_type].donor_name.value_counts().median(), "range",
      P[P.l1_type].donor_name.value_counts().min(), "to", P[P.l1_type].donor_name.value_counts().max())

**Result: reproduced.** The dandiset holds 194 cells of P in the ten L1 t-types, with 96 LAMP5,
35 MC4R, 33 PAX6, 15 L1 VIP and 15 other, the counts of Table S2. The other 16 files of P are
cells of non-L1 t-types. The 250 and 71 counts need transcriptomes and reconstructions and are not
tested.

## fig2c-subclass-ephys-differences

> Human L1 subclasses differ in electrophysiology, notably in sag and initial firing rate
> (Kruskal-Wallis, FDR-corrected p < 1e-7).

As l1_subclass_distinctness.ipynb cells 8 and 10: features with more than 40% non-missing values
among l1_type cells of P; Kruskal-Wallis across the four core subclasses; Benjamini-Hochberg;
Dunn's post hoc (code's Mann-Whitney shown as a variant). Run on reading (a) (features computed
here, the paper's values for the others) and (b) (the paper's values).

In [ ]:
fig2c = {}
for kind, T in [("a", TA), ("b", TB)]:
    data = T[T.P & T.l1_type]
    feats40 = [c for c in fcols if data[c].notna().mean() > 0.4]
    sub = T[T.P & T.core_l1_type].copy()
    sub["homology_type"] = sub.homology_type.cat.remove_unused_categories()
    res, pairs = L.run_kw_dunn(sub, feats40, "homology_type", posthoc="dunn")
    res_mw, pairs_mw = L.run_kw_dunn(sub, feats40, "homology_type", posthoc="mw")
    fig2c[kind] = (res, pairs, pairs_mw, feats40, sub)
    print(f"Reading ({kind}): {len(feats40)} features, {len(sub)} cells", sub.homology_type.value_counts().to_dict())
    print(f"  features with FDR p < 0.05: {(res.pval_fdr < 0.05).sum()}, < 1e-7: {(res.pval_fdr < 1e-7).sum()}")
    print(res.loc[["sag", "first_isi_inv_hero"], ["KW_H", "pval", "pval_fdr", "epsilon2", "n"]].to_string())
    print("  ten smallest p:"); print(res.head(10)[["pval_fdr", "epsilon2"]].to_string())

In [ ]:
print("By donor: number of donors per subclass in the test set (cells come from",
      fig2c["a"][4].donor_name.nunique(), "donors)")
print(fig2c["a"][4].groupby("homology_type", observed=True).donor_name.nunique().to_string())
# Leave-one-donor-out check of the two features: largest FDR p over runs without each donor
sub = fig2c["a"][4]
worst = {}
for f in ["sag", "first_isi_inv_hero"]:
    ps = []
    for d in sub.donor_name.unique():
        s = sub[sub.donor_name != d].dropna(subset=[f])
        ps.append(stats.kruskal(*[g[f].values for _, g in s.groupby("homology_type", observed=True)]).pvalue)
    worst[f] = max(ps)
print("Largest uncorrected KW p when any one donor is left out:", {k: f"{v:.2g}" for k, v in worst.items()})

**Result: partially reproduced.** Sag differs across the four subclasses with FDR p = 6.6e-20
on the features computed here. The initial firing rate at rheobase + 40 pA also differs, with FDR
p = 4.7e-7, above the 1e-7 of the claim; with the paper's own per-cell values it is 1.7e-8, below
it. The criterion judges on the features computed here (O1, reading (a)), so the claim holds in
part. Leaving out any one donor keeps the uncorrected p of sag below 1e-18.

## fig2b-subclass-ephys-profiles

> LAMP5 cells show small sag and adaptation, PAX6 cells minimal sag and a high initial firing
> rate, while MC4R and L1 VIP cells show strong sag.

In [ ]:
for kind in ["a", "b"]:
    res, pairs, pairs_mw, feats40, sub = fig2c[kind]
    med = sub.groupby("homology_type", observed=True)[["sag", "first_isi_inv_hero"]].median()
    n = sub.groupby("homology_type", observed=True)[["sag", "first_isi_inv_hero"]].count()
    print(f"Reading ({kind}): medians (n)"); print(med.round(3).astype(str).add(" (").add(n.astype(str)).add(")").to_string())
    i = med.sag.loc[["MC4R", "L1 VIP"]].min() > med.sag.loc[["LAMP5", "PAX6"]].max()
    ii = med.sag.idxmin() == "PAX6"
    # (iii) as amended by the reader's answer (questions/000630.yaml): PAX6 above LAMP5 and MC4R
    iii = med.first_isi_inv_hero.loc["PAX6"] > med.first_isi_inv_hero.loc[["LAMP5", "MC4R"]].max()
    iii_before = med.first_isi_inv_hero.idxmax() == "PAX6"
    print(f"  (i) MC4R and L1 VIP sag above LAMP5 and PAX6: {i}; (ii) PAX6 lowest sag: {ii}; "
          f"(iii) PAX6 initial rate above LAMP5 and MC4R: {iii}")
    print(f"  criterion before the answer, (iii) PAX6 highest of the four: {iii_before}")
    for f in ["sag", "first_isi_inv_hero"]:
        print(f"  Dunn FDR p, {f}:", {k: f"{v:.2g}" for k, v in pairs[f].items()})
    print(f"  range of sag {sub.sag.min():.3f} to {sub.sag.max():.3f}; first_isi_inv_hero "
          f"{sub.first_isi_inv_hero.min():.1f} to {sub.first_isi_inv_hero.max():.1f} Hz")

In [ ]:
res, pairs, pairs_mw, feats40, sub = fig2c["a"]
fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, f, lab in zip(axs, ["sag", "first_isi_inv_hero"], ["sag ratio", "initial firing rate, rheobase + 40 pA (Hz)"]):
    groups = [g[f].dropna().values for _, g in sub.groupby("homology_type", observed=True)]
    ax.boxplot(groups, tick_labels=list(sub.homology_type.cat.categories), showfliers=True)
    ax.set_ylabel(lab)
plt.tight_layout(); plt.show()
print("All points are within the axes (matplotlib autoscale).")

**Result: reproduced.** Part (iii) of the criterion was amended by a reader (questions/000630.yaml)
from "PAX6 has the highest median initial firing rate" to "PAX6 above LAMP5 and MC4R". Parts (i)
and (ii) hold under both readings: the MC4R (0.336) and L1 VIP (0.277) medians of sag exceed those
of LAMP5 (0.116) and PAX6 (0.051), and PAX6 has the lowest. Part (iii) as amended holds under both
readings: PAX6 has a median initial firing rate of 49.3 Hz, above LAMP5 (31.8 to 32.1) and MC4R
(28.8 to 30.5), with Dunn FDR p of 3.4e-5 or less. L1 VIP is higher than PAX6 (54.6 Hz under both
readings; Dunn FDR p = 0.39 to 1), so the criterion before the answer, PAX6 highest of the four,
fails.

## figs3a-sag-adaptation

> Spike adaptation properties are strongly and inversely related to sag across human L1 cells
> (R2 = 0.74).

As cells 8, 16 and 17: cells of P with l1_type; OLS of upstroke_adapt_ratio on sag with HC3 errors.

In [ ]:
rows = []
for kind, T, col in [("a, third over first", TA, "upstroke_adapt_ratio"),
                     ("a, first over third", TA.assign(upstroke_adapt_ratio=computed["upstroke_adapt_ratio_1over3"].reindex(TA.index)), "upstroke_adapt_ratio"),
                     ("b, paper's values", TB, "upstroke_adapt_ratio")]:
    data = T[T.P & T.l1_type].dropna(subset=["sag", col])
    fit = smf.ols(f"{col} ~ sag", data=data).fit(cov_type="HC3")
    rows.append({"reading": kind, "n": int(fit.nobs), "R2": round(fit.rsquared, 3), "slope": round(fit.params["sag"], 3),
                 "p (HC3)": f"{fit.f_pvalue:.2g}", "ratio range": f"{data[col].min():.2f} to {data[col].max():.2f}"})
print(pd.DataFrame(rows).to_string(index=False))
data = TA[TA.P & TA.l1_type].dropna(subset=["sag", "upstroke_adapt_ratio"])
print("Per subclass (reading a), R2 within each:")
for k, g in data.groupby("homology_type", observed=True):
    if len(g) > 3:
        print(f"  {k}: n={len(g)}, R2={smf.ols('upstroke_adapt_ratio ~ sag', data=g).fit().rsquared:.2f}")
for kind, T in [("a", TA), ("b", TB)]:
    d = T[T.P & T.l1_type].groupby("t-type")[["sag", "upstroke_adapt_ratio"]].median()
    fit = smf.ols("upstroke_adapt_ratio ~ sag", data=d).fit()
    print(f"Variant, t-type medians ({kind}): {len(d)} t-types, R2 {fit.rsquared:.2f}, slope {fit.params['sag']:.2f}, p {fit.f_pvalue:.2g}")
plt.figure(figsize=(4, 3.5)); plt.scatter(data.sag, data.upstroke_adapt_ratio, s=6)
plt.xlabel("sag ratio"); plt.ylabel("upstroke adaptation ratio (3rd/1st)"); plt.show()

**Result: not reproduced.** The regression of the upstroke adaptation ratio (third over first
spike, the order that matches the paper's table) on sag gives R2 = 0.33 (p = 1.3e-12) on the
features computed here, and R2 = 0.36 (p = 3.3e-14) on the paper's own per-cell values, against
0.74 +/- 0.074. The slope is negative: cells with more sag have a lower ratio, so their upstroke
changes more from the first to the third spike, the opposite of the direction the criterion
states. On the medians of the ten t-types R2 is 0.28 to 0.31.**

## fig2c-lamp5-gradient

> Within the LAMP5 subclass, sag and initial firing rate decrease from the NMBR to the DBP to the
> LCP2 t-type.

As cell 59: LAMP5 cells of the strict L1 set, t-type coded LCP2 = 0, DBP = 1, NMBR = 2; Spearman
correlation of every feature whose name has no "3"; Benjamini-Hochberg. The code also includes
the Amsterdam and Szeged LAMP5 cells, which the dandiset does not hold.

In [ ]:
order = {"LAMP5 LCP2": 0, "LAMP5 DBP": 1, "LAMP5 NMBR": 2}
for kind, T in [("a", TA), ("b", TB)]:
    data = T[T.in_strict_set & (T.homology_type == "LAMP5")].assign(cluster_i=lambda d: d["t-type"].map(order))
    feats_no3 = [c for c in fcols if "3" not in c]
    res = L.run_correlations(data, feats_no3, "cluster_i", filter_dropouts=False)
    print(f"Reading ({kind}): {len(data)} LAMP5 cells", data["t-type"].value_counts().to_dict(),
          f"| {len(res)} features, {int((res.pval_fdr < 0.001).sum())} with FDR p < 0.001")
    print(res.loc[["sag", "first_isi_inv_hero"]].to_string())
    print("  medians by t-type:", data.groupby("t-type")[["sag", "first_isi_inv_hero"]].median().round(3).to_dict())
    print("  donors by t-type:", data.groupby("t-type").donor_name.nunique().to_dict())
d3 = sites[sites.in_strict_set & (sites.homology_type == "LAMP5")].assign(cluster_i=lambda d: d["t-type"].map(order))
res3 = L.run_correlations(d3, [c for c in fcols if "3" not in c and c in d3], "cluster_i", filter_dropouts=False)
print(f"Variant with the Amsterdam and Szeged LAMP5 cells as the code has them (their features from the paper's tables): "
      f"{len(d3)} cells", d3.collaborator.value_counts().to_dict())
print(res3.loc[["sag", "first_isi_inv_hero"]].to_string())

**Result: partially reproduced.** Both features decrease from NMBR to DBP to LCP2 in rank order
(positive Spearman rho with the code LCP2 = 0, DBP = 1, NMBR = 2). The initial firing rate meets
the claim's threshold (FDR p = 2.6e-5 here, 2.5e-5 on the paper's values); sag does not (FDR p =
0.022 here, 0.0020 on the paper's values, against 0.001). With the Amsterdam and Szeged LAMP5
cells added from the paper's tables, as the code includes them, sag has FDR p = 0.0039, still
above 0.001.**

## figs6-donor-condition-ephys

> No electrophysiological feature of human L1 cells differs with donor medical condition, sex or
> age after FDR correction.

A claim of absence, judged on the paper's tests (l1_metadata_effects.ipynb cells 50, 51, 99, 100,
108 to 110): cells of P with l1_type; features more than 40% non-missing; epileptic or not
(code's `epileptic()`) and sex by two-sided exact Mann-Whitney; age by Spearman with the dropout
filter; Benjamini-Hochberg within each. The table's `medical_conditions`, `donor_sex` and
`donor_age` stand in for the LIMS records the code reads.

In [ ]:
def epileptic(x):
    if pd.isna(x):
        return x
    return "epileptic" if "epilep" in x.lower() else "non-epileptic"

for kind, T in [("a", TA), ("b", TB)]:
    data = T[T.P & T.l1_type].copy()
    data["epileptic"] = data.medical_conditions.map(epileptic)
    data["sex_t"] = data.donor_sex
    data["age_years"] = data.donor_age.str.extract(r"(\d+)")[0].astype(float)
    feats40 = [c for c in fcols if data[c].notna().mean() > 0.4]
    print(f"Reading ({kind}): {len(data)} cells, {len(feats40)} features")
    print("  epileptic:", data.epileptic.value_counts(dropna=False).to_dict(), "| donors with a condition:",
          data.dropna(subset=["epileptic"]).donor_name.nunique(), "of", data.donor_name.nunique())
    print("  sex:", data.sex_t.value_counts(dropna=False).to_dict(), "| age known:", int(data.age_years.notna().sum()))
    for name, res in [("medical condition", L.run_twosamp(data, feats40, "epileptic")),
                      ("sex", L.run_twosamp(data, feats40, "sex_t"))]:
        print(f"  {name}: smallest p {res.pval_mw.min():.3g} ({res.pval_mw.idxmin()}, r={res.loc[res.pval_mw.idxmin(), 'mw_r']:.2f}), "
              f"smallest FDR p {res.pval_mw_fdr.min():.3g}, features with FDR p < 0.05: {int((res.pval_mw_fdr < 0.05).sum())}")
    res = L.run_correlations(data, feats40, "age_years", filter_dropouts=True)
    print(f"  age: {len(res)} features after the dropout filter; smallest p {res.pval.min():.3g} ({res.pval.idxmin()}, "
          f"rho={res.loc[res.pval.idxmin(), 'r']:.2f}), smallest FDR p {res.pval_fdr.min():.3g}, "
          f"features with FDR p < 0.05: {int((res.pval_fdr < 0.05).sum())}")

**Result: reproduced.** No feature differs with medical condition, sex or age after FDR
correction under either reading (smallest FDR p 0.19, 0.51 and 0.43 on the features computed
here). The medical condition is known for only 86 of the 194 cells in the public table (65
epileptic, 21 not), so this test has less power than the paper's, which read the conditions from
the Allen Institute's donor database.**

## figs6c-region-ephys

> Some electrophysiological features of human L1 cells, including input resistance, vary with
> brain region; sparsely sampled regions either match MTG or have larger input resistance.

As l1_metadata_effects.ipynb cells 15, 29 and 30. `input_resistance_ss` is not computed here, so
it comes from the paper's table under both readings.

In [ ]:
for kind, T in [("a", TA), ("b", TB)]:
    df = T[T.l1_type & (T.collaborator == "AIBS") & (T.failed_fx_long_squares == False)].copy()
    df.loc[df.donor_name == "H18.03.003", "structure"] = "CeG"
    df.loc[df.donor_name == "H22.26.404", "structure"] = "SFG"
    vc = df.structure.value_counts()
    data = df[df.structure.isin(vc[vc >= 5].index) & (df.structure != "TemL")].copy()
    data["structure"] = pd.Categorical(data.structure, categories=["ITG", "MTG", "MFG", "SFG"], ordered=True)
    feats50 = [c for c in fcols if data[c].notna().mean() > 0.5]
    res, pairs = L.run_kw_dunn(data, feats50, "structure")
    print(f"Reading ({kind}): regions {data.structure.value_counts().to_dict()} (all regions {vc.to_dict()})")
    print(f"  {len(feats50)} features, {int((res.pval_fdr < 0.05).sum())} with FDR p < 0.05:", list(res.index[res.pval_fdr < 0.05]))
    for f in ["input_resistance_ss", "sag_peak_t", "downstroke_rheo", "avg_rate_max"]:
        if f in res.index:
            print(f"  {f}: FDR p {res.loc[f, 'pval_fdr']:.3g}; medians", data.groupby("structure", observed=True)[f].median().round(3).to_dict(),
                  "; Dunn FDR p", {k: f"{v:.2g}" for k, v in pairs.get(f, {}).items()})
    print("  donors per region:", data.groupby("structure", observed=True).donor_name.nunique().to_dict())

**Result: reproduced.** Over ITG, MTG, MFG and SFG, input resistance (`input_resistance_ss`,
the paper's values under both readings) varies with region (FDR p = 0.029), as do sag_peak_t,
downstroke_rheo and avg_rate_max (FDR p 0.016 to 0.029). No region has a median input resistance
below MTG's at Dunn FDR p < 0.05: SFG is lower (222 against 234 MOhm) with p = 0.13, and MFG is
higher (367 MOhm, p = 0.0073). MFG holds 5 cells from 3 donors.**

## fig4b-mc4r-sag-irregularity

> MC4R (rosehip) cells are distinguished from the rest of human L1 by strong sag and by irregular
> firing (Mann-Whitney, FDR p < 1e-4).

As l1_rosehip.ipynb cell 6: AIBS cells of the strict L1 set, group against all others; exact
Mann-Whitney per feature over all features; Benjamini-Hochberg. The irregularity ratio
(`ir_late_mean`) is custom code that is not public, so it comes from the paper's table (O1).

In [ ]:
for kind, T in [("a", TA), ("b", TB)]:
    base = T[T.in_strict_set & (T.collaborator == "AIBS")]
    for grp_name, mask in [("MC4R subclass", base.homology_type == "MC4R"),
                           ("SST CHRNA4", base["t-type"] == "SST CHRNA4"),
                           ("ADARB2 MC4R", base["t-type"] == "ADARB2 MC4R")]:
        df = base.assign(in_type=mask.map({True: "in", False: "rest"}))
        res = L.run_twosamp(df, fcols, "in_type")
        med = df.groupby("in_type")[["sag", "ir_late_mean"]].median()
        out = {f: f"FDR p {res.loc[f, 'pval_mw_fdr']:.2g}, median in {med.loc['in', f]:.3f} vs rest {med.loc['rest', f]:.3f}"
               for f in ["sag", "ir_late_mean"]}
        print(f"({kind}) {grp_name} (n={int(mask.sum())} of {len(df)}):", out)

**Result: reproduced.** Under reading (a) of the group, the MC4R subclass against the rest of
the Seattle strict L1 cells, sag (median 0.336 against 0.118, FDR p = 1.6e-18) and the irregularity
ratio (0.232 against 0.135, FDR p = 1.2e-5) are higher, both below 1e-4. Under reading (b) the
ADARB2 MC4R t-type alone meets it (FDR p 3.1e-17 and 4.2e-6); SST CHRNA4 alone does not. The
irregularity ratio is the paper's value under both feature readings.**

## fig4b-adarb2-irregular-firing

> Both rosehip t-types have strong sag, but only ADARB2 MC4R fires irregularly, with stronger and
> faster sag than SST CHRNA4.

As l1_rosehip.ipynb cell 9: cells of P grouped as LAMP5, SST CHRNA4, ADARB2 MC4R, PAX6, L1 VIP;
Dunn's tests on the seven pairs, Benjamini-Hochberg across them. `ir_late_mean`, `sag_peak_t` and
`sag_tau` come from the paper's table under both readings (not computed here).

In [ ]:
pairs7 = [(x, y) for x in ["LAMP5", "PAX6", "L1 VIP"] for y in ["SST CHRNA4", "ADARB2 MC4R"]] + [("SST CHRNA4", "ADARB2 MC4R")]
for kind, T in [("a", TA), ("b", TB)]:
    df = T[T.P & T.l1_type & (T.homology_type != "other")].copy()
    df["grp"] = np.where(df.homology_type == "MC4R", df["t-type"], df.homology_type.astype(str))
    df["grp"] = pd.Categorical(df.grp, categories=["LAMP5", "SST CHRNA4", "ADARB2 MC4R", "PAX6", "L1 VIP"])
    med = df.groupby("grp", observed=True)[["ir_late_mean", "sag", "sag_peak_t", "sag_tau"]].median()
    print(f"Reading ({kind}): n", df.grp.value_counts(sort=False).to_dict()); print(med.round(4).to_string())
    P7 = {f: L.dunn_pairs(df, f, "grp", pairs7) for f in ["ir_late_mean", "sag", "sag_peak_t", "sag_tau"]}
    print(pd.DataFrame(P7).map(lambda v: f"{v:.2g}").to_string())
    kw = {f: stats.kruskal(*[g[f].dropna() for _, g in df.groupby("grp", observed=True)]).pvalue for f in P7}
    print("  KW p:", {k: f"{v:.2g}" for k, v in kw.items()})
    i = (med.loc["ADARB2 MC4R", "ir_late_mean"] > med.loc["SST CHRNA4", "ir_late_mean"]) and P7["ir_late_mean"]["SST CHRNA4 - ADARB2 MC4R"] < 0.05
    ii = not any((med.loc["SST CHRNA4", "ir_late_mean"] > med.loc[x, "ir_late_mean"]) and P7["ir_late_mean"][f"{x} - SST CHRNA4"] < 0.05
                 for x in ["LAMP5", "PAX6", "L1 VIP"])
    iii = all(med.loc[t, "sag"] > med.loc[["LAMP5", "PAX6"], "sag"].max() for t in ["SST CHRNA4", "ADARB2 MC4R"])
    iv_a = med.loc["ADARB2 MC4R", "sag"] > med.loc["SST CHRNA4", "sag"] and med.loc["ADARB2 MC4R", "sag_peak_t"] < med.loc["SST CHRNA4", "sag_peak_t"]
    iv_b = med.loc["ADARB2 MC4R", "sag"] > med.loc["SST CHRNA4", "sag"] and med.loc["ADARB2 MC4R", "sag_tau"] < med.loc["SST CHRNA4", "sag_tau"]
    print(f"  (i) {i}  (ii) {ii}  (iii) {iii}  (iv) with sag_peak_t {iv_a}, with sag_tau {iv_b}")

**Result: partially reproduced.** Both MC4R t-types have a median sag above LAMP5 and PAX6
(iii), and ADARB2 MC4R has a stronger and faster sag than SST CHRNA4 under both readings of
"faster" (iv). ADARB2 MC4R has a much higher median irregularity ratio than SST CHRNA4 (0.421
against 0.153), but with 9 SST CHRNA4 cells the Dunn FDR p is 0.18, so (i) fails; and SST CHRNA4
is above LAMP5 (0.153 against 0.114) with Dunn FDR p = 0.046, so (ii) fails narrowly. The claim
holds for sag and for the direction of the irregularity difference, not for its significance.**

## fig5b-tnfaip8l3-ephys

> PAX6 TNFAIP8L3 cells are distinguished from all other human L1 cells by a high initial firing
> rate at rheobase and an after-depolarization following the last spike.

As l1_tnfaip.ipynb cells 10, 12 and 14: cells of P, PAX6 TNFAIP8L3 against all others; features
more than 50% non-missing among TNFAIP8L3 cells, plus adp_deltav; exact Mann-Whitney;
Benjamini-Hochberg. `postap_slope_rheo` ("ADP slope") and `adp_v_last_rheo` come from the paper's
table (not computed here).

In [ ]:
for kind, T in [("a", TA), ("b", TB)]:
    df = T[T.P].copy()
    df["adp_deltav"] = df.adp_v_last_rheo - df.threshold_v_rheo
    df["in_type"] = np.where(df["t-type"] == "PAX6 TNFAIP8L3", "in", "rest")
    inn = df[df.in_type == "in"]
    fl = [c for c in fcols if inn[c].notna().mean() > 0.5] + ["adp_deltav"]
    res = L.run_twosamp(df, fl, "in_type")
    print(f"Reading ({kind}): {len(inn)} TNFAIP8L3 cells against {int((df.in_type == 'rest').sum())}; {len(fl)} features")
    for f in ["first_isi_inv_rheo", "postap_slope_rheo", "adp_deltav"]:
        if f in res.index:
            print(f"  {f}: TNFAIP8L3 values {inn[f].round(2).tolist()}, rest median {df.loc[df.in_type == 'rest', f].median():.2f} "
                  f"(n={df.loc[df.in_type == 'rest', f].notna().sum()}), p {res.loc[f, 'pval_mw']:.2g}, FDR p {res.loc[f, 'pval_mw_fdr']:.2g}")
        else:
            print(f"  {f}: not tested (fewer than half the TNFAIP8L3 cells have it)")
    print("  smallest FDR p over features:", f"{res.pval_mw_fdr.min():.2g}")

**Result: reproduced.** The two PAX6 TNFAIP8L3 cells have initial firing rates at rheobase of
136 and 123 Hz (computed here, equal to the paper's values) against a median of 6.5 Hz in the
other cells (FDR p = 0.0081), and ADP slopes of 2.4 and 2.85 V/s (the paper's values) against
-0.43 (FDR p = 0.0051); both are below 0.01 and higher in TNFAIP8L3. The ADP voltage variant
(adp_deltav) has FDR p = 0.07.**

## fig6a-ephys-dprime-subclass-structure

> Pairwise electrophysiological distinctness (d') of human L1 t-types is smaller within subclasses
> than between them; the only within-subclass pair with high d' is LAMP5 NMBR against LAMP5 LCP2.

As l1_dprime.ipynb cells 9 to 13 and cluster_pairs.py: cells of P; t-types with at least 5 cells
(code) or 4 (legend); features more than 80% non-missing; preprocessing and random forest d' as in
the helper. Cells of P outside the L1 t-types take part in the preprocessing, as in the code. Seeds
0, 1 and 2 (a standard choice; the code sets none).

In [ ]:
within = {"LAMP5": ["LAMP5 NMBR", "LAMP5 DBP", "LAMP5 LCP2"], "MC4R": ["SST CHRNA4", "ADARB2 MC4R"],
          "PAX6": ["PAX6 CDH12", "PAX6 TNFAIP8L3"]}
sub_of = {t: s for s, ts in within.items() for t in ts}
dp_results = {}
tt_all = TA[TA.P]["t-type"].where(TA[TA.P]["t-type"].isin(L.HOMOLOGY)).value_counts()
print("L1 t-types in P:", tt_all.to_dict())
print("t-types kept with at least 5 cells:", list(tt_all[tt_all >= 5].index), "| with at least 4:", list(tt_all[tt_all >= 4].index))
same_sets = set(tt_all[tt_all >= 5].index) == set(tt_all[tt_all >= 4].index)
print("The two rules keep the same t-types:", same_sets, "(so the legend's rule is run only when they differ)")
for kind, T in [("a", TA), ("b", TB)]:
    for min_n in ([5] if same_sets else [5, 4]):
        d = T[T.P].copy()
        d["tt"] = d["t-type"].where(d["t-type"].isin(L.HOMOLOGY))
        vc = d.tt.value_counts()
        d = d[d.tt.isin(vc[vc >= min_n].index) | d.tt.isna()]
        if d.tt.isna().sum() < 5:
            d = d[d.tt.notna()]
        f80 = [c for c in fcols if d[c].notna().mean() > 0.8]
        for seed in [0, 1, 2]:
            D = L.pairwise_dprime(d, f80, "tt", seed)
            rows = []
            for i, a in enumerate(D.index):
                for b in D.index[:i]:
                    same = sub_of.get(a) is not None and sub_of.get(a) == sub_of.get(b)
                    rows.append({"a": a, "b": b, "within": same, "dprime": D.loc[a, b]})
            pr = pd.DataFrame(rows)
            dp_results[(kind, min_n, seed)] = (D, pr, len(f80), d.tt.value_counts().to_dict())
for key, (D, pr, nf, n) in dp_results.items():
    med_between = pr[~pr.within].dprime.median()
    hi_a = pr[pr.within & (pr.dprime >= 2.2)]
    hi_b = pr[pr.within & (pr.dprime > med_between)]
    print(f"reading {key[0]}, min {key[1]} cells, seed {key[2]}: {nf} features; mean within {pr[pr.within].dprime.mean():.2f}, "
          f"between {pr[~pr.within].dprime.mean():.2f} (median {med_between:.2f}); high within (>=2.2): "
          f"{[a + ' / ' + b for a, b in zip(hi_a.a, hi_a.b)]}; above median between: {[a + ' / ' + b for a, b in zip(hi_b.a, hi_b.b)]}")
print("\nt-types (min 5):", dp_results[("a", 5, 0)][3])
print("\nWithin-subclass d', reading a, min 5, seed 0:")
print(dp_results[("a", 5, 0)][1].query("within").round(2).to_string(index=False))
print("\nd' matrix, reading a, min 5, seed 0:"); print(dp_results[("a", 5, 0)][0].round(2).to_string())

**Result: reproduced, resting on an assumption.** Mean d' within subclasses (1.24 to 1.31)
is about half that between them (2.48 to 2.54) on the features computed here, for seeds 0 to 2.
With "high" read as d' of at least 2.2, the Methods' threshold for grouping t-types, the only high
within-subclass pair is LAMP5 NMBR against LAMP5 LCP2 (2.39 for seed 0) for every seed; on the
paper's values it is so for seed 0 and no within-subclass pair is high for seeds 1 and 2. With
"high" read as above the median between-subclass d' (2.40 to 2.62), no within-subclass pair is
high, NMBR against LCP2 included. The readings differ; the 2.2 threshold is adopted, as the one the
paper defines. Both t-type rules (at least 5 or 4 cells) keep the same nine t-types.**

## fig6d-gmm-subclass-f1

> Two-cluster Gaussian mixtures on the best pair of electrophysiological features recover each
> subclass with F1 scores of 0.81 (LAMP5), 0.69 (MC4R), 0.89 (PAX6) and 0.5 (all others).

As clustering_classification.ipynb cells 16 to 21 and clustering.py: AIBS cells with l1_type and
soma in L1 (`layer == '1'`); features more than 60% non-missing; for each group every single
feature and every pair, a 2-component Gaussian mixture (k-means init, 5 inits, seed 0 before each
fit as the code sets), the component holding most of the group's cells taken as the group; the
best F1. Preprocessing O3: (b) code, Yeo-Johnson and KNN imputation; (a) Methods, cells missing
more than 60% of features left out, then RobustScaler, KNN, LocalOutlierFactor < -20 set missing,
KNN.

In [ ]:
from itertools import combinations
from joblib import Parallel, delayed
from sklearn.impute import KNNImputer
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer

def gmm_f1(X, y):
    np.random.seed(0)
    pred = GaussianMixture(n_components=2, init_params="kmeans", n_init=5).fit_predict(X)
    i_true = int(np.mean(pred[y]) > 0.5)
    return f1_score(y, pred == i_true)

def best_subspace(Xall, names, y):
    sets = [(i,) for i in range(len(names))] + list(combinations(range(len(names)), 2))
    scores = Parallel(n_jobs=-1)(delayed(gmm_f1)(Xall[:, list(s)], y) for s in sets)
    k = int(np.argmax(scores))
    return scores[k], ", ".join(names[i] for i in sets[k]), len(sets)

gmm_out = {}
t0 = time.time()
for prep in ["b", "a"]:
    T = TA
    d = T[(T.collaborator == "AIBS") & T.l1_type & (T.failed_fx_long_squares == False) & (T.layer == "1")].copy()
    f60 = [c for c in fcols if d[c].notna().mean() > 0.6]
    if prep == "a":
        d = d[d[f60].notna().mean(axis=1) >= 0.4]
        X = L.preproc_pipeline().fit_transform(d[f60].values)
    else:
        X = Pipeline([("norm", PowerTransformer()), ("impute", KNNImputer())]).fit_transform(d[f60])
    grp = d.homology_type.astype(str).replace({"other": "all others", "L1 VIP": "all others"})
    for g in ["LAMP5", "MC4R", "PAX6", "all others"]:
        y = (grp == g).values
        gmm_out[(prep, g)] = best_subspace(X, f60, y) + (int(y.sum()), len(d))
print(f"GMM search took {time.time() - t0:.0f} s")
exp = {"LAMP5": 0.81, "MC4R": 0.69, "PAX6": 0.89, "all others": 0.5}
for (prep, g), (f1, fs, nsets, ng, n) in gmm_out.items():
    print(f"preprocessing ({prep}) {g}: n {ng} of {n}, best F1 {f1:.3f} (expected {exp[g]} +/- {exp[g] / 10:.3f}) "
          f"on [{fs}] over {nsets} subspaces; within tolerance: {abs(f1 - exp[g]) <= exp[g] / 10 + 1e-9}")

**Result: reproduced.** Under both preprocessing readings the best one- or two-feature
mixture gives F1 within tolerance for every group: LAMP5 0.806 and 0.785, MC4R 0.703 and 0.709,
PAX6 0.892 and 0.896, all others 0.526 and 0.490 (code and Methods preprocessing). Several of the
best features (adaptation ratios, sag_tau, sag_peak_t, ramp and short square features) are the
paper's values, as they are not computed here.**

## figs8a-classifier-cv

> A multi-class classifier assigns human L1 subclasses from electrophysiology with 82%
> cross-validated accuracy, balanced across subclasses.

As clustering_classification.ipynb cells 26 to 35 and 44 to 45: cells of the strict L1 set in the
four core subclasses with soma in L1, AIBS; logistic regression with balanced class weights after
RobustScaler and KNN imputation (O3 (a) adds the LocalOutlierFactor step); repeated stratified
5-fold cross-validation, 10 repeats (seed 0). Features (a) more than 50% non-missing at every site,
(b) the site-aligned subset. Choosing features by site needs the Amsterdam and Szeged feature
tables of the paper's code (tamas_features_E.csv, mansvelder_features_E.csv); they are used only
for this choice.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import RobustScaler

inh = sites[sites.in_strict_set & sites.homology_type.isin(L.SUBCLASSES) & (sites.failed_fx_long_squares == False) & (sites.layer == "1")]
print("Cells by site (strict set, core subclasses, soma in L1):", inh.collaborator.value_counts().to_dict())
common = [c for c in fcols if inh.groupby("collaborator")[c].apply(lambda x: x.notna().mean()).min() > 0.5]
site_fit = []
for c in common:
    fit = smf.ols(f"{c} ~ collaborator", data=inh).fit(cov_type="HC3")
    site_fit.append({"feature": c, "rsquared": fit.rsquared, "p": fit.f_pvalue})
site_fit = pd.DataFrame(site_fit).set_index("feature")
aligned = list(site_fit[(site_fit.rsquared < 0.05) & (site_fit.p > 0.05)].index)
print(f"features (a) common to every site: {len(common)}; (b) site-aligned: {len(aligned)}")

def cv_scores(X, y, outliers):
    steps = [("norm", RobustScaler()), ("impute", KNNImputer())]
    if outliers:
        steps = [("pre", L.preproc_pipeline())]
    pipe = Pipeline(steps + [("class", LogisticRegression(class_weight="balanced"))])
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
    acc, cms = [], []
    for tr, te in cv.split(X, y):
        pipe.fit(X[tr], y[tr]); pred = pipe.predict(X[te])
        acc.append(np.mean(pred == y[te])); cms.append(confusion_matrix(y[te], pred, labels=range(4)))
    cm = np.mean(cms, axis=0); cm = cm / cm.sum(axis=1, keepdims=True)
    return np.mean(acc), np.std(acc), np.mean(np.diag(cm)), cm

data = inh[inh.collaborator == "AIBS"]
y = data.homology_type.cat.codes.values
print("AIBS cells:", len(data), data.homology_type.value_counts(sort=False).to_dict())
for fname, fl in [("a common", common), ("b aligned", aligned)]:
    for prep, outl in [("code", False), ("Methods + LOF", True)]:
        acc, sd, bal, cm = cv_scores(data[fl].values, y, outl)
        print(f"features {fname}, preprocessing {prep}: accuracy {100 * acc:.1f}% (sd {100 * sd:.1f}), balanced {100 * bal:.1f}%; "
              f"per class recall {np.round(np.diag(cm), 2).tolist()}")
print("chance (largest class):", round(100 * pd.Series(y).value_counts(normalize=True).max(), 1), "%; balanced chance 25%")

**Result: reproduced.** Cross-validated accuracy is 84.6% with the features common to all
sites and the code's preprocessing, within 82 +/- 8.2; the other readings give 78.8 to 83.5%,
also within it. Balanced accuracy is 74.5 to 78.8%. The largest class is 52.4% of cells.**

## Summary

| Claim | Outcome |
|---|---|
| dandiset-cell-link | reproduced |
| patchseq-counts | reproduced |
| fig2c-subclass-ephys-differences | partially reproduced |
| fig2b-subclass-ephys-profiles | reproduced |
| figs3a-sag-adaptation | not reproduced |
| fig2c-lamp5-gradient | partially reproduced |
| figs6-donor-condition-ephys | reproduced |
| figs6c-region-ephys | reproduced |
| fig4b-mc4r-sag-irregularity | reproduced |
| fig4b-adarb2-irregular-firing | partially reproduced |
| fig5b-tnfaip8l3-ephys | reproduced |
| fig6a-ephys-dprime-subclass-structure | reproduced (assumption) |
| fig6d-gmm-subclass-f1 | reproduced |
| figs8a-classifier-cv | reproduced |
| figs8b-classifier-holdout | blocked, data missing |

The mouse claims are in `mouse_comparison.ipynb` and four morphology claims in
`morphology_tables.ipynb`; the record of every claim is in `attempts/000630-chartrand-2023-01.yaml`.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the features are computed here with IPFX 2.1.2 from the
long square sweeps, with the Methods' sweep QC as far as the files allow, and agree with the
paper's table within 1% for 71 to 98% of cells; the custom features the paper did not release, and
the adaptation ratios, which agree less, are the paper's values. Cell labels come
from the paper's table.

What was found about the dataset: it holds exactly the paper's primary set of 210 Seattle cells,
one per file, linked to the paper's table only through the LIMS number in `session_id`; the files
carry no t-type, subclass, layer or region, 6 files carry a time stamp in place of the cell name,
and the 62 Amsterdam and Szeged cells of the paper are not in it. Unacquired samples of sweeps that
stopped early are stored as zeros, and `bias_current` is stored in pA under a schema that fixes
amperes.

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min")